# DeFi Data Pipeline — FinTech 590
Fetches top Uniswap V3 pools, verifies contracts on Etherscan, and saves results.

## 0. Install Dependencies

In [4]:
import subprocess, sys

def ensure(*packages):
    for pkg in packages:
        try:
            __import__(pkg.replace("-", "_"))
        except ImportError:
            print(f"[setup] Installing {pkg}...")
            subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

ensure("requests", "pandas", "python-dotenv")
print("All dependencies ready.")

[setup] Installing python-dotenv...
All dependencies ready.


## 1. Config & API Key

In [5]:
import os, json, time, pathlib
import requests
import pandas as pd
from dotenv import load_dotenv

load_dotenv()
ETHERSCAN_API_KEY = os.getenv("ETHERSCAN_API_KEY")

if not ETHERSCAN_API_KEY:
    ETHERSCAN_API_KEY = input("Etherscan API key not found in .env — please paste it now: ").strip()
    if not ETHERSCAN_API_KEY:
        raise ValueError("Etherscan API key is required.")
    with open(".env", "a") as f:
        f.write(f"\nETHERSCAN_API_KEY={ETHERSCAN_API_KEY}\n")
    print("[setup] Key saved to .env for future runs.")
else:
    print("Etherscan API key loaded from .env.")

DEFILLAMA_POOLS  = "https://yields.llama.fi/pools"
ETHERSCAN_API    = "https://api.etherscan.io/api"
RATE_LIMIT_DELAY = 0.21   # 5 req/s free tier
MIN_TVL_USD      = 1_000_000
TOP_N            = 20

ABI_DIR  = pathlib.Path("pool_abis")
CSV_PATH = pathlib.Path("top_pools.csv")
ABI_DIR.mkdir(exist_ok=True)

print(f"Output: {CSV_PATH}  |  ABIs: {ABI_DIR}/")

Etherscan API key loaded from .env.
Output: top_pools.csv  |  ABIs: pool_abis/


## 2. Fetch Top Uniswap V3 Pools from DeFiLlama

> **Note:** The Graph's free hosted service was shut down in 2024.  
> This step now uses [DeFiLlama's free Yields API](https://yields.llama.fi/docs) — no API key required.

In [6]:
print("Fetching pool data from DeFiLlama...")
resp = requests.get(DEFILLAMA_POOLS, timeout=30)
resp.raise_for_status()
all_pools = resp.json()["data"]
print(f"  Total pools across all protocols: {len(all_pools):,}")

# Filter: Uniswap V3 on Ethereum, TVL > $1M
uni_pools = [
    p for p in all_pools
    if p.get("project") == "uniswap-v3"
    and p.get("chain") == "Ethereum"
    and (p.get("tvlUsd") or 0) >= MIN_TVL_USD
]
uni_pools.sort(key=lambda x: x.get("tvlUsd", 0), reverse=True)
top_raw = uni_pools[:TOP_N]
print(f"  Uniswap V3 / Ethereum / TVL>$1M: {len(uni_pools)} found, keeping top {len(top_raw)}\n")

def parse_fee_tier(pool_meta: str | None) -> int:
    """Convert poolMeta string like '0.05%' to Uniswap fee pips (500)."""
    try:
        return int(float(pool_meta.replace("%", "")) * 10_000)
    except (AttributeError, ValueError):
        return 0

pools = []
for p in top_raw:
    symbol_parts = p.get("symbol", "-").split("-")
    pools.append({
        "address":    p["pool"],
        "token0":     symbol_parts[0] if len(symbol_parts) > 0 else "",
        "token1":     symbol_parts[1] if len(symbol_parts) > 1 else "",
        "fee_tier":   parse_fee_tier(p.get("poolMeta")),
        "tvl_usd":    float(p.get("tvlUsd") or 0),
        "volume_usd": float(p.get("volumeUsd1d") or 0),
    })

for i, p in enumerate(pools, 1):
    print(f"  {i:>2}. {p['token0']}/{p['token1']} "
          f"(fee {p['fee_tier']/1e4:.2f}%)  "
          f"TVL=${p['tvl_usd']:>14,.0f}  "
          f"Vol(1d)=${p['volume_usd']:>12,.0f}")

Fetching pool data from DeFiLlama...
  Total pools across all protocols: 18,173
  Uniswap V3 / Ethereum / TVL>$1M: 134 found, keeping top 20

   1. USDC/WETH (fee 0.05%)  TVL=$    95,189,976  Vol(1d)=$ 170,865,141
   2. WETH/USDT (fee 0.30%)  TVL=$    64,298,333  Vol(1d)=$  13,704,711
   3. WBTC/WETH (fee 0.05%)  TVL=$    46,020,167  Vol(1d)=$  27,902,080
   4. WBTC/WETH (fee 0.30%)  TVL=$    44,982,493  Vol(1d)=$   3,207,818
   5. WBTC/CBBTC (fee 0.01%)  TVL=$    29,652,526  Vol(1d)=$   5,158,775
   6. WBTC/USDC (fee 0.30%)  TVL=$    27,163,381  Vol(1d)=$   2,080,198
   7. WBTC/USDT (fee 0.30%)  TVL=$    26,404,287  Vol(1d)=$   1,608,704
   8. USDC/USDT (fee 0.01%)  TVL=$    26,343,758  Vol(1d)=$  11,233,357
   9. AUSD/USDC (fee 0.01%)  TVL=$    25,013,642  Vol(1d)=$   3,411,795
  10. USDC/WETH (fee 0.30%)  TVL=$    22,470,572  Vol(1d)=$   2,996,931
  11. LINK/WETH (fee 0.30%)  TVL=$    22,340,227  Vol(1d)=$   1,199,784
  12. USDM/USDT (fee 0.05%)  TVL=$    19,815,375  Vol(1d)=$      

## 3. Verify Each Pool Contract on Etherscan

In [7]:
def etherscan_get_abi(address: str) -> tuple[bool, list | None]:
    """Returns (is_verified, abi_or_None). Handles rate-limit with retry."""
    params = {
        "module":  "contract",
        "action":  "getabi",
        "address": address,
        "apikey":  ETHERSCAN_API_KEY,
    }
    for attempt in range(3):
        r = requests.get(ETHERSCAN_API, params=params, timeout=15)
        r.raise_for_status()
        payload = r.json()
        if payload["status"] == "1":
            return True, json.loads(payload["result"])
        if "rate limit" in payload.get("result", "").lower():
            print(f"    [rate-limit] backing off 2 s ...")
            time.sleep(2)
            continue
        return False, None
    return False, None


for p in pools:
    addr = p["address"]
    print(f"  Checking {addr[:10]}...  ({p['token0']}/{p['token1']})", end="", flush=True)
    verified, abi = etherscan_get_abi(addr)
    p["etherscan_verified"] = verified

    if verified and abi:
        abi_path = ABI_DIR / f"{addr}.json"
        abi_path.write_text(json.dumps(abi, indent=2))
        print(f"  ✓ verified  (ABI saved → pool_abis/{addr}.json)")
    else:
        print(f"  ✗ not verified")

    time.sleep(RATE_LIMIT_DELAY)

  Checking 665dc8bc-c...  (USDC/WETH)  ✗ not verified
  Checking fc9f488e-8...  (WETH/USDT)  ✗ not verified
  Checking d59a5728-d...  (WBTC/WETH)  ✗ not verified
  Checking c5599b3a-e...  (WBTC/WETH)  ✗ not verified
  Checking c0bbcf6c-9...  (WBTC/CBBTC)  ✗ not verified
  Checking bbecbf69-a...  (WBTC/USDC)  ✗ not verified
  Checking 2608e751-4...  (WBTC/USDT)  ✗ not verified
  Checking e737d721-f...  (USDC/USDT)  ✗ not verified
  Checking 458a64c5-2...  (AUSD/USDC)  ✗ not verified
  Checking 49717ee2-9...  (USDC/WETH)  ✗ not verified
  Checking 3025b6b3-e...  (LINK/WETH)  ✗ not verified
  Checking eb21dc22-0...  (USDM/USDT)  ✗ not verified
  Checking 2f340114-f...  (WM/USDC)  ✗ not verified
  Checking 3dd2c646-c...  (WBTC/USDT)  ✗ not verified
  Checking b4ef32d6-0...  (TBTC/WBTC)  ✗ not verified
  Checking a9ee1b5f-5...  (UNI/WETH)  ✗ not verified
  Checking 84fb00e5-3...  (WETH/WEETH)  ✗ not verified
  Checking be43dcf6-3...  (USDT/USDF)  ✗ not verified
  Checking 40769df5-7...  (WB

## 4. Save Results

In [8]:
df = pd.DataFrame(pools, columns=[
    "address", "token0", "token1", "fee_tier",
    "tvl_usd", "volume_usd", "etherscan_verified"
])
df.to_csv(CSV_PATH, index=False)

verified_count = df["etherscan_verified"].sum()
abi_files      = list(ABI_DIR.glob("*.json"))

print(f"top_pools.csv  → {len(df)} rows saved")
print(f"pool_abis/     → {len(abi_files)} ABI files ({verified_count} verified pools)")
print()
df[["token0", "token1", "fee_tier", "tvl_usd", "etherscan_verified"]]

top_pools.csv  → 20 rows saved
pool_abis/     → 0 ABI files (0 verified pools)



,token0,token1,fee_tier,tvl_usd,etherscan_verified
0,USDC,WETH,500,95189976.0,False
1,WETH,USDT,3000,64298333.0,False
2,WBTC,WETH,500,46020167.0,False
3,WBTC,WETH,3000,44982493.0,False
4,WBTC,CBBTC,100,29652526.0,False
5,WBTC,USDC,3000,27163381.0,False
6,WBTC,USDT,3000,26404287.0,False
7,USDC,USDT,100,26343758.0,False
8,AUSD,USDC,100,25013642.0,False
9,USDC,WETH,3000,22470572.0,False
